# Student Performance Prediction

**Goal:** predict math exam scores using background and preparation variables. This notebook follows the project handbook: cleaning, EDA, feature engineering, 80/20 split, Linear Regression, R²/MAE/RMSE, and coefficient interpretation.

> Dataset: Students Performance in Exams (Kaggle). If the CSV is missing, run `python ../download_data.py` from the project root or use the download cell below.

In [1]:
from pathlib import Path
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import joblib


In [2]:
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_PATH = ROOT / "data" / "StudentsPerformance.csv"
DATA_PATH.parent.mkdir(parents=True, exist_ok=True)
if not DATA_PATH.exists():
    url = "https://raw.githubusercontent.com/NadavKiani/Students-Performance-in-Exams/master/StudentsPerformance.csv"
    urllib.request.urlretrieve(url, DATA_PATH)
df = pd.read_csv(DATA_PATH)
df.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75


## 1. Inspect and clean the data

In [3]:
print("Shape:", df.shape)
print("\nInfo:")
df.info()
print("\nMissing values:")
display(df.isna().sum())
print("Duplicate rows:", df.duplicated().sum())

Shape: (1000, 8)

Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB

Missing values:


gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

Duplicate rows: 0


In [4]:
df.columns = [c.strip().lower().replace("/", "_").replace(" ", "_") for c in df.columns]
score_cols = ["math_score", "reading_score", "writing_score"]
cat_cols = [c for c in df.columns if c not in score_cols]
df = df.drop_duplicates().copy()
for col in cat_cols:
    df[col] = df[col].astype("string").str.strip().str.lower()
for col in score_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=score_cols + cat_cols)
for col in score_cols:
    df = df[df[col].between(0, 100)]
print("Cleaned shape:", df.shape)
display(df.head())

Cleaned shape: (1000, 8)


,gender,race_ethnicity,parental_level_of_education,lunch,test_preparation_course,math_score,reading_score,writing_score
0,female,group b,bachelor's degree,standard,none,72,72,74
1,female,group c,some college,standard,completed,69,90,88
2,female,group b,master's degree,standard,none,90,95,93
3,male,group a,associate's degree,free/reduced,none,47,57,44
4,male,group c,some college,standard,none,76,78,75


In [5]:
print("Categorical value counts:")
for col in cat_cols:
    print(f"\n{col}:")
    display(df[col].value_counts())


Categorical value counts:

gender:


gender
female    518
male      482
Name: count, dtype: Int64


race_ethnicity:


race_ethnicity
group c    319
group d    262
group b    190
group e    140
group a     89
Name: count, dtype: Int64


parental_level_of_education:


parental_level_of_education
some college          226
associate's degree    222
high school           196
some high school      179
bachelor's degree     118
master's degree        59
Name: count, dtype: Int64


lunch:


lunch
standard        645
free/reduced    355
Name: count, dtype: Int64


test_preparation_course:


test_preparation_course
none         642
completed    358
Name: count, dtype: Int64

## 2. Exploratory data analysis

In [6]:
df[score_cols].describe().round(2)

,math_score,reading_score,writing_score
count,1000.00,1000.00,1000.00
mean,66.09,69.17,68.05
std,15.16,14.60,15.20
min,0.00,17.00,10.00
25%,57.00,59.00,57.75
50%,66.00,70.00,69.00
75%,77.00,79.00,79.00
max,100.00,100.00,100.00


In [7]:
sns.set_theme(style="whitegrid")
df[score_cols].melt(var_name="subject", value_name="score").pipe(lambda d: sns.histplot(data=d, x="score", hue="subject", element="step", stat="density", common_norm=False))
plt.title("Score distributions by subject"); plt.show()

<Figure size 640x480 with 1 Axes>

In [8]:
sns.barplot(data=df, x="gender", y="math_score", errorbar=None, color="steelblue")
plt.title("Average math score by gender"); plt.show()

<Figure size 640x480 with 1 Axes>

In [9]:
sns.barplot(data=df, x="test_preparation_course", y="math_score", errorbar=None, color="steelblue")
plt.title("Average math score by test preparation"); plt.show()

<Figure size 640x480 with 1 Axes>

In [10]:
plt.figure(figsize=(10,5))
sns.barplot(data=df, x="parental_level_of_education", y="math_score", errorbar=None, color="steelblue")
plt.xticks(rotation=25, ha="right"); plt.title("Average math score by parental education"); plt.show()

<Figure size 1000x500 with 1 Axes>

In [11]:
sns.heatmap(df[score_cols].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, fmt=".2f")
plt.title("Correlation between subject scores"); plt.show()

<Figure size 640x480 with 2 Axes>

In [12]:
sns.boxplot(data=df, x="lunch", y="math_score")
plt.title("Math score by lunch type"); plt.show()

<Figure size 640x480 with 1 Axes>

## 3. Feature engineering and model training
We create an average-score feature for analysis and a binary test-preparation flag. For prediction, reading/writing scores are intentionally excluded because the goal is to predict math score from background/preparation information rather than from other exam results.

In [13]:
df["average_score"] = df[score_cols].mean(axis=1)
df["test_prep_completed"] = (df["test_preparation_course"] == "completed").astype(int)
X = df[["gender", "race_ethnicity", "parental_level_of_education", "lunch", "test_preparation_course"]]
y = df["math_score"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)
preprocessor = ColumnTransformer([("categorical", OneHotEncoder(handle_unknown="ignore", drop="first"), X.columns.tolist())])
model = Pipeline([("preprocessor", preprocessor), ("regressor", LinearRegression())])
model.fit(X_train, y_train)
pred = model.predict(X_test)

## 4. Evaluation

In [14]:
metrics = {"R2": r2_score(y_test, pred), "MAE": mean_absolute_error(y_test, pred), "RMSE": np.sqrt(mean_squared_error(y_test, pred))}
pd.DataFrame([metrics]).round(3)

,R2,MAE,RMSE
0,0.176,11.27,14.16


In [15]:
sns.scatterplot(x=y_test, y=pred, alpha=.7)
plt.plot([0,100],[0,100], linestyle="--", color="red")
plt.xlim(0,100); plt.ylim(0,100); plt.xlabel("Actual math score"); plt.ylabel("Predicted math score")
plt.title("Actual vs predicted math scores"); plt.show()

<Figure size 640x480 with 1 Axes>

## 5. Top factors from coefficients
Coefficients are interpreted relative to the omitted reference category for each categorical feature. They indicate association in this fitted model, not causation.

In [16]:
feature_names = model.named_steps["preprocessor"].get_feature_names_out()
coef_df = pd.DataFrame({"feature": feature_names, "coefficient": model.named_steps["regressor"].coef_})
coef_df["abs_coefficient"] = coef_df["coefficient"].abs()
coef_df.sort_values("abs_coefficient", ascending=False).head(5)

,feature,coefficient,abs_coefficient
10,categorical__lunch_standard,11.523997,11.523997
4,categorical__race_ethnicity_group e,9.077935,9.077935
11,categorical__test_preparation_course_none,-5.874513,5.874513
0,categorical__gender_male,4.520714,4.520714
6,categorical__parental_level_of_education_high ...,-4.090503,4.090503


## 6. Correlation analysis of top factors

Categorical predictors are one-hot encoded, then each category indicator is correlated with math score. These are associations, not causal effects.


In [17]:
correlation_features = pd.get_dummies(X, drop_first=False, dtype=int)
correlation_df = correlation_features.copy()
correlation_df["math_score"] = y.to_numpy()
math_correlations = correlation_df.corr(numeric_only=True)["math_score"].drop("math_score")
math_correlations = math_correlations.reindex(math_correlations.abs().sort_values(ascending=False).index)
top_correlations = math_correlations.head(5).rename("correlation_with_math_score").reset_index()
top_correlations = top_correlations.rename(columns={"index": "factor"})
display(top_correlations.round(3))

plt.figure(figsize=(9,5))
plot_corr = math_correlations.head(5).sort_values()
plt.barh(plot_corr.index, plot_corr.values, color="steelblue")
plt.axvline(0, color="black", linewidth=0.8)
plt.xlabel("Correlation with math score")
plt.ylabel("Encoded factor")
plt.title("Top 5 categorical factors associated with math score")
plt.tight_layout(); plt.show()


,factor,correlation_with_math_score
0,lunch_free/reduced,-0.351
1,lunch_standard,0.351
2,race_ethnicity_group e,0.206
3,test_preparation_course_completed,0.178
4,test_preparation_course_none,-0.178


<Figure size 900x500 with 1 Axes>

## 7. Save outputs

In [18]:
OUT = ROOT / "outputs"
FIG = OUT / "figures"
OUT.mkdir(exist_ok=True); FIG.mkdir(exist_ok=True)
df.to_csv(OUT / "cleaned_student_performance.csv", index=False)
coef_df.sort_values("abs_coefficient", ascending=False).to_csv(OUT / "model_coefficients.csv", index=False)
joblib.dump(model, OUT / "linear_regression_model.joblib")
print("Saved cleaned data, coefficients, and model to", OUT)

Saved cleaned data, coefficients, and model to /mnt/data/final_project/Student-Performance-Prediction/outputs
